# 1. Consulta 

Para mejorar la clasificación de estados cognitivos en sistemas de Interfaz Cerebro-Computadora (BCI) basados en imaginación motora, se seleccionaron tres índices adicionales a la Densidad Espectral de Potencia (PSD) calculada en el Proyecto 1. Estos índices abordan las bioseñales desde diferentes perspectivas, tanto temporales como estadísticas.

---
---

## 1.1. Parámetros de Hjorth (Movilidad y Complejidad)
* **Descripción de la medida:** Parametros introducidos por Bo Hjorth en 1970. Estos parametros son descriptores estadísticos calculados directamente en el dominio del tiempo a partir de la varianza de la señal y sus derivadas, lo que evita transformaciones frecuenciales de alto costo computacional.

    * **Movilidad:** Representa la frecuencia media de la señal y se calcula como la raíz cuadrada de la razón entre la varianza de la primera derivada y la varianza de la señal original.
    * **Complejidad:** Mide el grado de similitud de la señal con una onda senoidal pura, evaluando los cambios en el ancho de banda.

* **Uso en los datos del proyecto:** Durante la imaginación de movimiento de las manos izquierda/derecha, se genera una Desincronización Relacionada con Eventos (ERD) en los ritmos Mu y Beta sobre la corteza motora. La *Movilidad* capturará vectorialmente este desplazamiento hacia frecuencias más altas en los canales $C3$ y $C4$ de manera inmediata.


**Referencia:** Hjorth, B. (1970). *EEG analysis based on time domain properties*. Electroencephalography and Clinical Neurophysiology, 29(3), 306-310.


---

## 1.2. Entropía de Permutación (Permutation Entropy - PermEn)
* **Descripción de la medida:** Es una métrica basada en la teoría del caos que cuantifica el grado de regularidad, predictibilidad y desorden de una serie temporal. Analiza el orden secuencial (patrones ordinales) en el que aparecen los puntos contiguos de la señal de EEG en lugar de evaluar únicamente sus amplitudes físicas.

* **Uso en los datos del proyecto:** En estado de reposo, el EEG presenta ritmos altamente síncronos y predecibles (baja entropía). Al iniciar la imaginación motora activa, el reclutamiento neuronal desincroniza la red cortical local, lo que incrementa el caos y la complejidad de la señal. La Entropía de Permutación discriminará el reposo de las tareas activas midiendo este quiebre de predictibilidad.


**Referencia:** Bandt, C., & Pompe, B. (2002). *Permutation entropy: a natural complexity measure for time series*. Physical Review Letters, 88(17), 174102.

---

## 1.3. Índice de Asimetría de Potencia Hemisférica (Brain Symmetry Index - ASI)
* **Descripción de la medida:** Es una métrica neurofisiológica cuantitativa que evalúa la diferencia de potencia espectral entre canales homólogos ubicados de forma simétrica en ambos hemisferios cerebrales (izquierdo vs. derecho). Su formulación matemática estándar es:

$$ASI = \frac{P_{\text{izq}} - P_{\text{der}}}{P_{\text{izq}} + P_{\text{der}}}$$

* **Uso en los datos del proyecto:** Debido a la organización contralateral del sistema motor, la imaginación de movimiento de la **mano derecha** atenuará la potencia (ERD) en el canal $C3$ (hemisferio izquierdo), mientras que la de la **mano izquierda** atenuará la potencia en $C4$ (hemisferio derecho). Calculando el $ASI$ específicamente entre $C3$ y $C4$ para las bandas Mu y Beta, se obtendrán valores característicos sesgados positivamente para una condición y negativamente para la otra, maximizando la eficiencia de los clasificadores de Machine Learning.


**Referencia:** van Putten, M. J. (2007). *The brain symmetry index*. Journal of Clinical Neurophysiology, 24(4), 365-370.


# 2. Plan de Análisis 

El objetivo de esta metodología es establecer un flujo de trabajo riguroso que abarque desde el preprocesamiento de bioseñales de EEG hasta el entrenamiento y evaluación de modelos de Machine Learning, garantizando un formato de datos óptimo y el cumplimiento estricto de las buenas prácticas en ciencia de datos.

---

## 2.1. Procesamiento Base y Segmentación (Epoching)
A partir de los registros de EEG crudos (109 sujetos), se aplicará un pipeline de limpieza centrado en la corteza motora:
1. **Selección de canales:** Se aislarán principalmente los canales **C3** (hemisferio izquierdo) y **C4** (hemisferio derecho), encargados de registrar la actividad motora contralateral.
2. **Filtrado digital:** Se aplicará un filtro pasa-banda Butterworth IIR de fase cero (rango de 0.5 Hz a 40 Hz) para eliminar la deriva de la línea base y el ruido de alta frecuencia, preservando la integridad temporal de las ondas, sin embargo, tambien es posible usar un pasa-altas a 0.5 Hz (atenuación de deriva basal) y un pasa-bajas a 40 Hz (eliminación de ruido muscular y de red).
3. **Segmentación y Etiquetado:** Se extraerán ventanas de tiempo (épocas) de la señal continua basadas en los marcadores de eventos. Para asegurar un conjunto de datos balanceado, se extraerán **aproximadamente 20 épocas por sujeto y por condición**. La variable objetivo (`Etiqueta`) se codificará numéricamente:
   * `0`: Reposo (T0 / Baseline)
   * `1`: Tarea 1 (T1 - Imaginación de mano izquierda)
   * `2`: Tarea 2 (T2 - Imaginación de mano derecha)

## 2.2. Extracción Vectorial de Características
Para cada época segmentada, se calcularán índices avanzados empleando optimización vectorial con `NumPy` :
1. **Potencia Espectral (PSD):** Método de Welch para extraer la energía en las bandas Mu (8-12 Hz) y Beta (13-30 Hz).
2. **Parámetros de Hjorth:** Movilidad y Complejidad, para evaluar cambios rápidos en la frecuencia media de la señal.
3. **Entropía de Permutación (PermEn):** Cuantificación del grado de caos temporal y desincronización de la señal.
4. **Índice de Asimetría (ASI):** Contraste matemático inter-hemisférico de las potencias entre $C3$ y $C4$.

## 2.3. Estructuración de la Base de Datos (DataFrame)
Todos los índices extraídos se consolidarán de forma organizada en un único objeto `pandas.DataFrame`. Cada fila representará un ensayo (época), resultando en aproximadamente 60 filas por sujeto. Todos los modelos predictivos consumirán exclusivamente las columnas de características numéricas bajo la siguiente estructura tabular:

| Sujeto |  PSD_Mu_C3 | PSD_Beta_C4 | Hjorth_Mov_C3 | PermEn_C4 | ASI_Mu |Etiqueta (0,1,2)|
| :---: | :---: | :---: | :---: | :---: | :---: | :---: |
| sub-001 |---| --- |---|---|---|`0`|
| sub-001 |---| ---|---|---|---|`1`|
| sub-002 |---|---|---|---|---|`2`|

## 2.4. Exploración Estadística, Correlación y Normalización
Antes de la fase de modelado, los datos pasarán por una etapa estricta de validación y transformación:
* **Descripción Estadística:** Se aplicará estadística descriptiva para revisar medias, varianzas y verificar que la base de datos no contenga valores atípicos que puedan sesgar el modelo.
* **Matriz de Correlación:** Se evaluará la colinealidad entre características (variables independientes) mediante un mapa de calor. Las características altamente correlacionadas (redundantes) podrán ser descartadas.
* **Normalización:** Dado que los algoritmos de Machine Learning y Redes Neuronales son sensibles a variables con magnitudes dispares, todos los valores de las características serán normalizados (ej. Escalado Estándar Z-score, llevando las variables a media 0 y varianza 1) antes de ingresar a los modelos.

## 2.5. Entrenamiento y Evaluación de Modelos (Clasificación)
Se construirá el pipeline de aprendizaje automático estructurado de la siguiente manera:
1. **División de Datos (Train/Test Split):** Partición del conjunto de datos normalizados para entrenamiento (ej. 80%) y pruebas (ej. 20%).
2. **Modelado:** Entrenamiento de algoritmos tradicionales como Máquinas de Soporte Vectorial (SVM) y Extreme Gradient Boosting (XGBoost), además de la definición y comparación de tres arquitecturas diferentes de Redes Neuronales (variando su profundidad y parámetros).
3. **Evaluación de Predicciones:** Se contrastarán las etiquetas originales (`Etiqueta`) frente a las etiquetas predichas por el modelo, tanto en el conjunto de entrenamiento (para diagnosticar posible subajuste) como en el de test (para evitar el sobreajuste).
4. **Criterios de Éxito:** El desempeño de los algoritmos se medirá principalmente mediante las métricas de **Accuracy (Exactitud)** y **F1-Score**. Se buscará que estos valores sean cercanos a `1.0`, lo que indicará un aprendizaje real de los patrones fisiológicos; valores cercanos a `0.0` implicarán nula capacidad predictiva. Adicionalmente, se analizarán las **matrices de confusión** para estudiar los errores específicos de clasificación entre tareas.

In [14]:
import numpy as np
import pandas as pd
from scipy.signal import welch
import antropy as ant #Para la entropía, se usa la librería antropy
import mne
import os
from scipy.signal import butter, sosfiltfilt




#En la función de Hjorth se usa np.var y np.diff pasándoles el parámetro axis=-1. Esto significa que se opera sobre toda la dimensión del tiempo al instante

def calcular_hjorth_vectorial(x, axis=-1):
    """
    Calcula la Movilidad y Complejidad de Hjorth de forma puramente vectorial.
    x: Matriz de datos (ej. [epocas, canales, tiempo] o [epocas, tiempo])
    """
    # Varianza de la señal original
    var_x = np.var(x, axis=axis)
    
    # Primera derivada (diferencia finita) y su varianza
    dx = np.diff(x, axis=axis)
    var_dx = np.var(dx, axis=axis)
    
    # Segunda derivada y su varianza
    ddx = np.diff(dx, axis=axis)
    var_ddx = np.var(ddx, axis=axis)
    
    # Cálculo de los parámetros
    movilidad = np.sqrt(var_dx / var_x)
    movilidad_dx = np.sqrt(var_ddx / var_dx)
    complejidad = movilidad_dx / movilidad
    
    return movilidad, complejidad

def calcular_psd_bandas(x, sfreq, bandas, axis=-1):
    """
    Calcula la potencia espectral absoluta usando el método de Welch y la integra 
    para bandas específicas de manera vectorial.
    """
    # Calculamos la PSD para todas las épocas de un golpe
    freqs, psd = welch(x, fs=sfreq, nperseg=int(sfreq), axis=axis)
    
    potencias = {}
    for nombre_banda, (fmin, fmax) in bandas.items():
        # Encontrar los índices de las frecuencias de interés
        idx = np.logical_and(freqs >= fmin, freqs <= fmax)
        # Integrar el área bajo la curva en la banda (usando np.trapz vectorizado)
        potencia_banda = np.trapz(psd[..., idx], freqs[idx], axis=axis)
        potencias[nombre_banda] = potencia_banda
        
    return potencias

def calcular_asi(potencia_c3, potencia_c4):
    """
    Calcula el Índice de Asimetría (ASI) entre C3 y C4 de forma matricial.
    """
    return (potencia_c3 - potencia_c4) / (potencia_c3 + potencia_c4)

def calcular_entropia_permutacion(x, order=3, delay=1):
    """
    Aplica la Entropía de Permutación de 'antropy' a través de todas las épocas.
    Como antropy requiere 1D, usamos np.apply_along_axis para optimizarlo.
    """
    return np.apply_along_axis(lambda sig: ant.perm_entropy(sig, order=order, delay=delay), axis=-1, arr=x)

In [15]:
RUNS_BASELINE = [1, 2]
RUNS_TAREA1   = [3, 7, 11]   # movimiento real mano izq/der
RUNS_TAREA2   = [4, 8, 12]   # imaginación mano izq/der
RUNS_INTERES  = RUNS_BASELINE + RUNS_TAREA1 + RUNS_TAREA2  # [1,2,3,4,7,8,11,12]

SUJETOS = list(range(1, 110))  # 109 sujetos

DATA_PATH = r"C:\Users\ASUS\Desktop\Sujetos_Proyecto"

ruta_ejemplo = f'{DATA_PATH}/sub-001/eeg/sub-001_task-motion_run-4_eeg.set'
raw_ejemplo  = mne.io.read_raw_eeglab(ruta_ejemplo, preload=True)

print(raw_ejemplo)
print(raw_ejemplo.info)
print(raw_ejemplo.ch_names)
print(raw_ejemplo.annotations)

ruta_ejemplo = f'{DATA_PATH}/sub-004/eeg/sub-004_task-motion_run-4_eeg.set'
raw_ejemplo  = mne.io.read_raw_eeglab(ruta_ejemplo, preload=True)

print(raw_ejemplo)
print(raw_ejemplo.info)
print(raw_ejemplo.ch_names)
print(raw_ejemplo.annotations)

<RawEEGLAB | sub-001_task-motion_run-4_eeg.set, 64 x 20000 (125.0 s), ~9.8 MiB, data loaded>
<Info | 8 non-empty values
 bads: []
 ch_names: Fc5, Fc3, Fc1, Fcz, Fc2, Fc4, Fc6, C5, C3, C1, Cz, C2, C4, C6, ...
 chs: 64 EEG
 custom_ref_applied: False
 dig: 67 items (3 Cardinal, 64 EEG)
 highpass: 0.0 Hz
 lowpass: 80.0 Hz
 meas_date: unspecified
 nchan: 64
 projs: []
 sfreq: 160.0 Hz
>
['Fc5', 'Fc3', 'Fc1', 'Fcz', 'Fc2', 'Fc4', 'Fc6', 'C5', 'C3', 'C1', 'Cz', 'C2', 'C4', 'C6', 'Cp5', 'Cp3', 'Cp1', 'Cpz', 'Cp2', 'Cp4', 'Cp6', 'Fp1', 'Fpz', 'Fp2', 'Af7', 'Af3', 'Afz', 'Af4', 'Af8', 'F7', 'F5', 'F3', 'F1', 'Fz', 'F2', 'F4', 'F6', 'F8', 'Ft7', 'Ft8', 'T7', 'T8', 'T9', 'T10', 'Tp7', 'Tp8', 'P7', 'P5', 'P3', 'P1', 'Pz', 'P2', 'P4', 'P6', 'P8', 'Po7', 'Po3', 'Poz', 'Po4', 'Po8', 'O1', 'Oz', 'O2', 'Iz']
<Annotations | 30 segments: TASK2T0 (15), TASK2T1 (8), TASK2T2 (7)>
<RawEEGLAB | sub-004_task-motion_run-4_eeg.set, 64 x 19680 (123.0 s), ~9.7 MiB, data loaded>
<Info | 8 non-empty values
 bads: []


In [25]:
DATA_PATH = r"C:\Users\ASUS\Desktop\Sujetos_Proyecto"
EXCLUSIONES = {89: [1, 2, 3]}
FS = 160.0
TMIN = 0.0
TMAX = 4.0
CANALES_INTERES = ['Fz', 'Fc3', 'Fc1', 'Fc4', 'C5', 'C3', 'Cz', 'C4', 
                   'Cp3', 'Cp1', 'Cpz', 'Cp2', 'Cp4', 'P1', 'P2', 'Poz']


def load_raw(sujeto_id, run_id, data_path=DATA_PATH):
    """Carga un archivo .set de EEGLAB con MNE."""
    ruta = os.path.join(data_path, sujeto_id, 'eeg', f'{sujeto_id}_task-motion_run-{run_id}_eeg.set')
    if not os.path.exists(ruta):
        print(f'  [AVISO] Archivo no encontrado: {ruta}')
        return None
    return mne.io.read_raw_eeglab(ruta, preload=True, verbose=False)

def filter_signal(data, fs):
    """Filtro IIR Butterworth de tu proyecto anterior."""
    f_hp, f_lp, orden = 0.5, 40.0, 7 
    sos = butter(orden, [f_hp, f_lp], btype='bandpass', output='sos', fs=fs)
    return sosfiltfilt(sos, data, axis=-1)

def epoch_signal(raw, run_id, sujeto_id, canales=CANALES_INTERES, fs=FS, tmin=TMIN, tmax=TMAX):
    """Segmenta la señal EEG en épocas por condición."""
    presentes = [c for c in canales if c in raw.ch_names]
    n_muestras_epoca = round((tmax - tmin) * fs)
    data_filt = filter_signal(raw.copy().pick(presentes).get_data(), fs=fs)
    
    if run_id in RUNS_BASELINE:
        return {'baseline': data_filt[np.newaxis, :, :]}
        
    prefijo = 'TASK1' if run_id in RUNS_TAREA1 else 'TASK2' if run_id in RUNS_TAREA2 else None
    if not prefijo: return {}

    onsets_izq = [ann['onset'] for ann in raw.annotations if ann['description'] == f'{prefijo}T1']
    onsets_der = [ann['onset'] for ann in raw.annotations if ann['description'] == f'{prefijo}T2']
    
    def _extraer_epocas(onsets):
        if not onsets: return None
        n_total = data_filt.shape[1]
        idx_ini = (np.array(onsets) * fs).astype(int)
        idx_fin = idx_ini + n_muestras_epoca
        mascara = idx_fin <= n_total
        if not mascara.any(): return None
        idx_ini = idx_ini[mascara]
        indices = idx_ini[:, None] + np.arange(n_muestras_epoca)
        return data_filt[:, indices].transpose(1, 0, 2)

    epocas = {}
    arr_izq = _extraer_epocas(onsets_izq)
    arr_der = _extraer_epocas(onsets_der)
    if arr_izq is not None: epocas['izq'] = arr_izq
    if arr_der is not None: epocas['der'] = arr_der
    return epocas

In [26]:
# Definición de las bandas de interés
bandas_eeg = {'Mu': (8, 12), 'Beta': (13, 30)}

# Lista vacía para guardar las filas del dataset
dataset_rows = []
fs = FS  

# Mapeo de etiquetas 
etiquetas_map = {
    'baseline': 0, # Reposo T0
    'imag_izq': 1, # Imaginación Izquierda T1
    'imag_der': 2  # Imaginación Derecha T2
}

#Prueba con 5 sujetos
sujetos = [f"sub-{i:03d}" for i in range(1, 6)] 

for sujeto in sujetos:
    runs_excl = EXCLUSIONES.get(int(sujeto[-3:]), [])
    
    #1. EXTRAER REPOSO (BASELINE) 
    epocas_reposo = []
    for run_id in [1, 2]: 
        if run_id in runs_excl: continue
        raw = load_raw(sujeto, run_id)
        if raw is None: continue
        ep = epoch_signal(raw, run_id, sujeto)
        if 'baseline' in ep:
            epocas_reposo.append(ep['baseline'])
            
    #2. EXTRAER IMAGINACIÓN MOTORA
    epocas_imag_izq = []
    epocas_imag_der = []
    for run_id in [4, 8, 12]: # Runs correspondientes a Imaginación
        if run_id in runs_excl: continue
        raw = load_raw(sujeto, run_id)
        if raw is None: continue
        ep = epoch_signal(raw, run_id, sujeto)
        if 'izq' in ep:
            epocas_imag_izq.append(ep['izq'])
        if 'der' in ep:
            epocas_imag_der.append(ep['der'])

    # Verificacion de extracción de épocas
    if not epocas_reposo or not epocas_imag_izq or not epocas_imag_der:
        print(f"[{sujeto}] Faltan datos, omitiendo...")
        continue 
        
    # Unión de las épocas de los diferentes runs en un solo bloque matricial
    data_reposo = np.concatenate(epocas_reposo, axis=0)
    data_im_izq = np.concatenate(epocas_imag_izq, axis=0)
    data_im_der = np.concatenate(epocas_imag_der, axis=0)
    
    # Diccionario para procesar en bucle
    condiciones = {
        0: data_reposo,
        1: data_im_izq,
        2: data_im_der
    }
    
    # Encontrar automáticamente los índices de C3 y C4 en la lista CANALES_INTERES
    idx_c3 = CANALES_INTERES.index('C3')
    idx_c4 = CANALES_INTERES.index('C4')
        
    for etiqueta, data_matriz in condiciones.items():
        # Extracción de la señal matricial específica para C3 y C4
        c3_data = data_matriz[:, idx_c3, :]
        c4_data = data_matriz[:, idx_c4, :]
        
        # 1. Calcular PSD Vectorial (Retorna diccionario con Mu y Beta)
        psd_c3 = calcular_psd_bandas(c3_data, fs, bandas_eeg)
        psd_c4 = calcular_psd_bandas(c4_data, fs, bandas_eeg)
        
        # 2. Calcular Asimetría (ASI)
        asi_mu = calcular_asi(psd_c3['Mu'], psd_c4['Mu'])
        asi_beta = calcular_asi(psd_c3['Beta'], psd_c4['Beta'])
        
        # 3. Calcular Hjorth (Movilidad y Complejidad) vectorizado
        mov_c3, comp_c3 = calcular_hjorth_vectorial(c3_data)
        mov_c4, comp_c4 = calcular_hjorth_vectorial(c4_data)
        
        # 4. Calcular Entropía de Permutación (PermEn)
        permen_c3 = calcular_entropia_permutacion(c3_data)
        permen_c4 = calcular_entropia_permutacion(c4_data)
        
        # 5. Agrupar características en filas y agregarlas al dataset
        n_epocas = data_matriz.shape[0]
        for i in range(n_epocas):
            fila = {
                'Sujeto': sujeto,
                'PSD_Mu_C3': psd_c3['Mu'][i],
                'PSD_Beta_C4': psd_c4['Beta'][i],
                'Hjorth_Mov_C3': mov_c3[i],
                'Hjorth_Mov_C4': mov_c4[i],
                'Hjorth_Comp_C3': comp_c3[i],
                'Hjorth_Comp_C4': comp_c4[i],
                'PermEn_C3': permen_c3[i],
                'PermEn_C4': permen_c4[i],
                'ASI_Mu': asi_mu[i],
                'ASI_Beta': asi_beta[i],
                'Etiqueta': etiqueta, # Aquí va el 0, 1 o 2

            }
            dataset_rows.append(fila)


df_caracteristicas = pd.DataFrame(dataset_rows)

print("\n¡DataFrame creado exitosamente!")
print(f"Total de filas: {df_caracteristicas.shape[0]}")
display(df_caracteristicas.head())


¡DataFrame creado exitosamente!
Total de filas: 235


,Sujeto,PSD_Mu_C3,PSD_Beta_C4,Hjorth_Mov_C3,Hjorth_Mov_C4,Hjorth_Comp_C3,Hjorth_Comp_C4,PermEn_C3,PermEn_C4,ASI_Mu,ASI_Beta,Etiqueta
0,sub-001,1.549599e-10,1.354348e-10,0.338317,0.325996,2.453939,2.579856,2.128385,2.118654,0.181758,0.166352,0
1,sub-001,5.087152e-10,1.796251e-10,0.386625,0.380907,1.995198,1.988819,2.072579,2.069019,0.074659,0.123337,0
2,sub-001,9.346727e-11,1.187769e-10,0.350482,0.332597,2.617001,2.795712,2.196860,2.236012,0.121307,0.135281,1
3,sub-001,1.161658e-10,1.221226e-10,0.272238,0.287939,3.260601,3.097683,2.178812,2.201467,0.271669,0.138052,1
4,sub-001,1.067915e-10,1.521642e-10,0.355413,0.328545,2.456089,2.708334,2.154233,2.197421,0.245410,0.164880,1
